# 01 — kNN donor selection in embedding space

For each treated site, the donors are the **10 nearest controls in before-period embedding
space**: Euclidean distance over all 768 dimensions, computed separately per sensor, pool =
all 260 controls, with replacement. No covariates, no land-cover filter — similarity is
defined entirely by the learned representation. (Per Jun's decision, 2026-08-14: "change the
donor to similarity based matching — this makes more sense for embedding-based features.")

## What we keep from the collaborator's workflow

Two things only — deliberately, so the feature-based estimator and the comparison to the feature-based donor sets stay valid:

1. **Ranked selection of the same number of donors.** Score every candidate, rank ascending,
   keep the 10 best, assign `control_rank` 1–10, require exactly 10 — the same 10-donor
   design as the feature-based pipeline. Holding the design fixed is what makes "the feature-based donors vs kNN donors"
   a controlled comparison of similarity definitions under the same estimator.
2. **The output format, unchanged.** The matching table uses the feature-based exact schema
   (`treatment_site_id, counterfactual_site_id, control_rank, pair_id, match_quality,
   matching_score`), and the notebook builds per-sensor base trees —
   `REAP/data/embeddings_knn_s1/` and `..._knn_s2/`, rasters symlinked, each with its own
   `site_matching_table.csv` — so `02_DiD_estimator_feature09.ipynb` consumes them via the
   `EMBED_DID_BASE` env var with zero changes.

Everything else is ours: we are doing a different operation (metric retrieval in a latent
space) on different objects (embedding vectors), so a line-by-line diff against the feature-based notebook
06 is not the right frame. (Reframed 2026-08-15, Jun's decision; code cells still carry the feature-based
function/variable names for readability of the history.)

## Design guarantees

1. **Before-period only.** The query (treated site) and every candidate vector are
   **before**-period embeddings. The after period — where the treatment effect lives —
   never enters donor selection, so matching cannot leak the outcome.
2. **Per sensor, separately.** S1 and S2 are different latent spaces (different encoders,
   no fusion): each treated site gets 10 S1-nearest and 10 S2-nearest donors — generally
   different sites — and distances are never compared across sensors, only within one space.


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/data/wang/junh/githubs/latent-synthetic-control")
EMB = ROOT / "REAP" / "data" / "embeddings"
ED = ROOT / "REAP" / "notebooks" / "embed_DiD"

CONTROLS_PER_TREATMENT = 10  # the feature-based §5 constant, unchanged

her_table = pd.read_csv(EMB / "finals" / "site_matching_table.csv")
treatments = sorted(her_table["treatment_site_id"].astype(str).unique())
controls = sorted(her_table["counterfactual_site_id"].astype(str).unique())
print(len(treatments), "treatments;", len(controls), "controls in the pool")

emb = pd.read_csv(EMB / "site_embeddings.csv")
ecols = [c for c in emb.columns if c.startswith("emb_")]
before = (emb.query("period == 'before'")
          .set_index(["sensor", "site_id"])[ecols])
print("before-period embeddings:", before.shape)

26 treatments; 260 controls in the pool


before-period embeddings: (572, 768)


## §15 (edited) — matching-score function

The feature-based structure kept verbatim: copy the pool, attach a `matching_score` column, lower =
better. EDIT A: the score is the Euclidean distance between before-period embeddings.

In [2]:
def calculate_matching_score(
    candidate_pool,
    treatment_row,
):
    """
    Calculate embedding-similarity matching differences.

    Lower scores represent better matches.

    EDIT A (vs the feature-based §15): elevation/slope/distance terms replaced by the
    Euclidean distance between before-period embedding vectors.
    """

    output = candidate_pool.copy()

    output["matching_score"] = np.sqrt(
        (
            (output[ecols].to_numpy() - treatment_row[ecols].to_numpy())
            ** 2
        ).sum(axis=1)
    )

    return output

## §16 (edited) — select 10 controls for each treatment site

The feature-based loop skeleton: per treatment, score the pool, sort ascending, take the 10 best,
assign `control_rank` 1–10, record the match. EDITs C–F applied (no NLCD filter, no
staged pools, with replacement, no spacing check).

In [3]:
knn_tables = {}
for sensor in ("sentinel1", "sentinel2"):
    matching_records = []
    incomplete_treatment_ids = []

    for treatment_id in treatments:
        treatment_row = before.loc[(sensor, treatment_id)]

        candidate_pool = before.loc[sensor].loc[controls].copy()  # EDIT B/E: full 260, with replacement

        candidate_pool = calculate_matching_score(
            candidate_pool,
            treatment_row,
        )

        ranked_pool = candidate_pool.sort_values("matching_score")

        if len(ranked_pool) < CONTROLS_PER_TREATMENT:
            print(f"{treatment_id} obtained only {len(ranked_pool)} of "
                  f"{CONTROLS_PER_TREATMENT} required controls.")
            incomplete_treatment_ids.append(treatment_id)
            continue

        for control_rank, (control_id, selected) in enumerate(
            ranked_pool.head(CONTROLS_PER_TREATMENT).iterrows(), start=1
        ):
            matching_records.append({
                "pair_id": f"knn_{sensor}_{treatment_id.replace('treatment_', '')}",
                "treatment_site_id": treatment_id,
                "counterfactual_site_id": control_id,
                "control_rank": control_rank,
                "match_quality": "embedding_knn",
                "matching_score": float(selected["matching_score"]),
            })

    matching_table = pd.DataFrame(matching_records)
    assert not incomplete_treatment_ids
    assert (matching_table.groupby("treatment_site_id").size() == CONTROLS_PER_TREATMENT).all()
    knn_tables[sensor] = matching_table
    print(f"{sensor}: {len(matching_table)} matches, "
          f"{matching_table['counterfactual_site_id'].nunique()} distinct donors used")

sentinel1: 260 matches, 122 distinct donors used


sentinel2: 260 matches, 130 distinct donors used


## Build the two parallel base trees (symlinked rasters + own matching table)

In [4]:
for sensor, tag in (("sentinel1", "s1"), ("sentinel2", "s2")):
    base = ROOT / "REAP" / "data" / f"embeddings_knn_{tag}"
    finals = base / "finals"
    finals.mkdir(parents=True, exist_ok=True)
    (base / "test").mkdir(exist_ok=True)
    for sub in ("sentinel1", "sentinel2"):
        link = finals / sub
        target = EMB / "finals" / sub
        if link.is_symlink() or link.exists():
            if link.is_symlink():
                link.unlink()
        link.symlink_to(target, target_is_directory=True)
    knn_tables[sensor].to_csv(finals / "site_matching_table.csv", index=False)
    print(f"{base}  <- donors by {sensor} similarity "
          f"(rasters symlinked; run 09 with EMBED_DID_BASE={base})")

/data/wang/junh/githubs/latent-synthetic-control/REAP/data/embeddings_knn_s1  <- donors by sentinel1 similarity (rasters symlinked; run 09 with EMBED_DID_BASE=/data/wang/junh/githubs/latent-synthetic-control/REAP/data/embeddings_knn_s1)
/data/wang/junh/githubs/latent-synthetic-control/REAP/data/embeddings_knn_s2  <- donors by sentinel2 similarity (rasters symlinked; run 09 with EMBED_DID_BASE=/data/wang/junh/githubs/latent-synthetic-control/REAP/data/embeddings_knn_s2)


## Diagnostics — how the embedding-chosen donors relate to the feature-based covariate-chosen ones

In [5]:
nlcd = {}
for _, r in her_table.iterrows():
    nlcd[str(r["treatment_site_id"])] = int(r["nlcd_class"])
    nlcd[str(r["counterfactual_site_id"])] = int(r["nlcd_class"])

hers_by_t = her_table.groupby("treatment_site_id")["counterfactual_site_id"].apply(
    lambda s: set(s.astype(str)))

for sensor in ("sentinel1", "sentinel2"):
    t = knn_tables[sensor]
    ours_by_t = t.groupby("treatment_site_id")["counterfactual_site_id"].apply(set)
    overlap = np.mean([len(ours_by_t[k] & hers_by_t[k]) for k in treatments])
    nlcd_agree = np.mean([nlcd[r["counterfactual_site_id"]] == nlcd[r["treatment_site_id"]]
                          for _, r in t.iterrows()])
    usage = t["counterfactual_site_id"].value_counts()
    print(f"{sensor}: mean overlap with the feature-based 10 donors = {overlap:.2f}/10 | "
          f"kNN donors sharing treated NLCD class = {100*nlcd_agree:.1f}% | "
          f"distinct donors used = {usage.size}/260, max reuse = {usage.max()}")

sentinel1: mean overlap with the feature-based 10 donors = 1.35/10 | kNN donors sharing treated NLCD class = 60.0% | distinct donors used = 122/260, max reuse = 6
sentinel2: mean overlap with the feature-based 10 donors = 1.00/10 | kNN donors sharing treated NLCD class = 61.9% | distinct donors used = 130/260, max reuse = 7
